In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
pd.set_option('display.max_columns', None)


In [2]:
 # Plot settings
plt.rcParams["figure.figsize"] = (12, 6)
sns.set(style="whitegrid")

raw = "../data/raw"

In [3]:
# Load all datasets
sales = pd.read_csv(f"{raw}/sunnybest_sales.csv", parse_dates=["date"])
products = pd.read_csv(f"{raw}/sunnybest_products.csv")
stores = pd.read_csv(f"{raw}/sunnybest_stores.csv")
calendar = pd.read_csv(f"{raw}/sunnybest_calendar.csv", parse_dates=["date"])
promos = pd.read_csv(f"{raw}/sunnybest_promotions.csv", parse_dates=["date"])
weather = pd.read_csv(f"{raw}/sunnybest_weather.csv", parse_dates=["date"])
inventory = pd.read_csv(f"{raw}/sunnybest_inventory.csv", parse_dates=["date"])


# surpress warning

# warnings.filterwarnings("ignore", category=DtypeWarning)

/var/folders/rt/0zxshr9s4g713_r6y5sjpqk80000gn/T/ipykernel_58080/3054379081.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(f"{raw}/sunnybest_sales.csv", parse_dates=["date"])


In [4]:
df = (
    sales
    .merge(products, on="product_id", how="left", suffixes=("", "_product"))
    .merge(stores, on="store_id", how="left", suffixes=("", "_store"))
    .merge(calendar, on="date", how="left", suffixes=("", "_cal"))
    .merge(weather, on=["date", "city"], how="left", suffixes=("", "_weather"))
    .merge(promos, on=["date", "store_id", "product_id"], how="left", suffixes=("", "_promo"))
)

print(df.shape)
df.head()


(1227240, 43)


,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
0,2021-01-01,1,1001,0,445838.0,445838,0,0,NaN,0.0,1,1,0,Benin,Large,Televisions,LG Televisions Model-120,Televisions,LG,445838,332772,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
1,2021-01-01,1,1002,2,500410.0,500410,0,0,NaN,1000820.0,6,4,0,Benin,Large,Mobile Phones,Tecno Mobile Model-199,Mobile Phones,Tecno,500410,346208,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
2,2021-01-01,1,1003,2,399365.0,399365,0,0,NaN,798730.0,2,0,1,Benin,Large,Mobile Phones,Tecno Mobile Model-905,Mobile Phones,Tecno,399365,244124,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
3,2021-01-01,1,1004,4,305796.0,305796,0,0,NaN,1223184.0,10,6,0,Benin,Large,Mobile Phones,Infinix Mobile Model-121,Mobile Phones,Infinix,305796,221242,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
4,2021-01-01,1,1005,5,462752.0,462752,0,0,NaN,2313760.0,28,23,0,Benin,Large,Mobile Phones,Samsung Mobile Model-781,Mobile Phones,Samsung,462752,326219,1,24,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN


## Stores Data Overview & Business Context

The stores dataset provides **static reference information** about each physical retail location.  
Each row represents **one store**, describing its geographical location and operational scale.  
This data does not change daily and is used to **add contextual information** to sales records rather than to record sales events themselves.

---

### Store Data Grain

- **One row = one store**
- The dataset grows only when:
  - a new store opens, or  
  - store attributes are updated

---

### Key Store Columns & Meanings

| Column | Business Meaning |
|------|------------------|
| store_id | Unique identifier for a physical retail store |
| city | City in which the store is located |
| store_size | Classification of store scale (e.g. Small, Medium, Large), reflecting capacity and expected footfall |

---

### How This Data Is Used

Store attributes are **joined onto the sales data** to help explain differences in demand and performance across locations.  
For example, larger stores or stores in certain cities may consistently sell higher volumes than others.

---

### Important Note

The stores dataset **does not record sales**.  
It provides **context** that helps interpret sales outcomes observed in the daily sales dataset.


In [5]:
df.head()

,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
0,2021-01-01,1,1001,0,445838.0,445838,0,0,NaN,0.0,1,1,0,Benin,Large,Televisions,LG Televisions Model-120,Televisions,LG,445838,332772,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
1,2021-01-01,1,1002,2,500410.0,500410,0,0,NaN,1000820.0,6,4,0,Benin,Large,Mobile Phones,Tecno Mobile Model-199,Mobile Phones,Tecno,500410,346208,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
2,2021-01-01,1,1003,2,399365.0,399365,0,0,NaN,798730.0,2,0,1,Benin,Large,Mobile Phones,Tecno Mobile Model-905,Mobile Phones,Tecno,399365,244124,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
3,2021-01-01,1,1004,4,305796.0,305796,0,0,NaN,1223184.0,10,6,0,Benin,Large,Mobile Phones,Infinix Mobile Model-121,Mobile Phones,Infinix,305796,221242,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
4,2021-01-01,1,1005,5,462752.0,462752,0,0,NaN,2313760.0,28,23,0,Benin,Large,Mobile Phones,Samsung Mobile Model-781,Mobile Phones,Samsung,462752,326219,1,24,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN


In [6]:
df.head()

,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
0,2021-01-01,1,1001,0,445838.0,445838,0,0,NaN,0.0,1,1,0,Benin,Large,Televisions,LG Televisions Model-120,Televisions,LG,445838,332772,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
1,2021-01-01,1,1002,2,500410.0,500410,0,0,NaN,1000820.0,6,4,0,Benin,Large,Mobile Phones,Tecno Mobile Model-199,Mobile Phones,Tecno,500410,346208,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
2,2021-01-01,1,1003,2,399365.0,399365,0,0,NaN,798730.0,2,0,1,Benin,Large,Mobile Phones,Tecno Mobile Model-905,Mobile Phones,Tecno,399365,244124,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
3,2021-01-01,1,1004,4,305796.0,305796,0,0,NaN,1223184.0,10,6,0,Benin,Large,Mobile Phones,Infinix Mobile Model-121,Mobile Phones,Infinix,305796,221242,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
4,2021-01-01,1,1005,5,462752.0,462752,0,0,NaN,2313760.0,28,23,0,Benin,Large,Mobile Phones,Samsung Mobile Model-781,Mobile Phones,Samsung,462752,326219,1,24,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN


In [7]:
df.shape

(1227240, 43)

In [8]:
df.describe(include='all')

,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
count,1227240,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,292,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,1227240,1227240,1227240,1227240,1227240,1227240,1.227240e+06,1.227240e+06,1.227240e+06,1.227240e+06,1227240,1227240,1227240,1227240,1227240,1227240,1.227240e+06,1.227240e+06,1.227240e+06,1227240,1227240,1227240,1227240,1227240,1.227240e+06,1.227240e+06,1227240,292,292.000000,292.0
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,7,3,9,120,9,26,NaN,NaN,NaN,NaN,7,7,6,3,3,3,NaN,NaN,NaN,7,2,2,2,3,NaN,NaN,3,4,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Discount,NaN,NaN,NaN,NaN,Benin,Small,Mobile Phones,LG Televisions Model-120,Mobile Phones,LG,NaN,NaN,NaN,NaN,SunnyBest Benin Main,Benin,Esan West,Edo Central,High Street,Small,NaN,NaN,NaN,Friday,False,False,False,Early Rainy,NaN,NaN,Rainy,Discount,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,171,NaN,NaN,NaN,NaN,175320,701280,235221,10227,235221,153405,NaN,NaN,NaN,NaN,175320,175320,350640,701280,701280,701280,NaN,NaN,NaN,175560,876120,1210440,1186920,514080,NaN,NaN,505080,171,NaN,NaN
mean,2022-12-31 23:59:59.999999744,4.000000e+00,1.060500e+03,2.435828e+00,3.124509e+05,3.124595e+05,2.978228e-03,2.379323e-04,NaN,2.561325e+05,8.220214e+00,5.784386e+00,4.094798e-02,NaN,NaN,NaN,NaN,NaN,NaN,3.124595e+05,2.177207e+05,5.083333e-01,1.350000e+01,NaN,NaN,NaN,NaN,NaN,NaN,2.022501e+03,6.522930e+00,1.572964e+01,NaN,NaN,NaN,NaN,NaN,2.849883e+01,3.412927e+00,NaN,NaN,12.517123,1.0
min,2021-01-01 00:00:00,1.000000e+00,1.001000e+03,0.000000e+00,6.900000e+02,6.900000e+02,0.000000e+00,0.000000e+00,NaN,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,6.900000e+02,4.560000e+02,0.000000e+00,6.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,2.021000e+03,1.000000e+00,1.000000e+00,NaN,NaN,NaN,NaN,NaN,2.210000e+01,0.000000e+00,NaN,NaN,0.000000,1.0
25%,2022-01-01 00:00:00,2.000000e+00,1.030750e+03,0.000000e+00,6.401200e+04,6.590800e+04,0.000000e+00,0.000000e+00,NaN,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,6.590800e+04,4.568850e+04,0.000000e+00,6.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,2.022000e+03,4.000000e+00,8.000000e+00,NaN,NaN,NaN,NaN,NaN,2.720000e+01,2.000000e-01,NaN,NaN,0.000000,1.0
50%,2023-01-01 00:00:00,4.000000e+00,1.060500e+03,1.000000e+00,2.514150e+05,2.521605e+05,0.000000e+00,0.000000e+00,NaN,1.126680e+05,2.000000e+00,1.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,2.521605e+05,1.772030e+05,1.000000e+00,1.200000e+01,NaN,NaN,NaN,NaN,NaN,NaN,2.023000e+03,7.000000e+00,1.600000e+01,NaN,NaN,NaN,NaN,NaN,2.850000e+01,2.200000e+00,NaN,NaN,10.000000,1.0
75%,2024-01-01 00:00:00,6.000000e+00,1.090250e+03,2.000000e+00,5.041590e+05,5.065822e+05,0.000000e+00,0.000000e+00,NaN,4.108960e+05,7.000000e+00,4.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,5.065822e+05,3.351362e+05,1.000000e+00,2.400000e+01,NaN,NaN,NaN,NaN,NaN,NaN,2.024000e+03,1.000000e+01,2.300000e+01,NaN,NaN,NaN,NaN,NaN,2.980000e+01,5.200000e+00,NaN,NaN,20.000000,1.0
max,2024-12-31 00:00:00,7.000000e+00,1.120000e+03,1.150000e+02,8.796500e+05,8.796500e+05,3.000000e+01,1.000000e+00,NaN,5.090272e+06,4.440000e+02,3.720000e+02,1.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,8.796500e+05,6.847270e+05,1.000000e+00,2.400000e+01,NaN,NaN,NaN,NaN,NaN,NaN,2.024000e+03,1.200000e+01,3.100000e+01,NaN,NaN,NaN,NaN,NaN,3.520000e+01,2.130000e+01,NaN,NaN,30.000000,1.0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1227240 entries, 0 to 1227239
Data columns (total 43 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   date                   1227240 non-null  datetime64[ns]
 1   store_id               1227240 non-null  int64         
 2   product_id             1227240 non-null  int64         
 3   units_sold             1227240 non-null  int64         
 4   price                  1227240 non-null  float64       
 5   regular_price          1227240 non-null  int64         
 6   discount_pct           1227240 non-null  int64         
 7   promo_flag             1227240 non-null  int64         
 8   promo_type             292 non-null      object        
 9   revenue                1227240 non-null  float64       
 10  starting_inventory     1227240 non-null  int64         
 11  ending_inventory       1227240 non-null  int64         
 12  stockout_occurred      12272

- We are planning inventory for the next 4 weeks. Can you forecast weekly sales for each store so we know how much stock to order?

In [10]:
df.head()

,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
0,2021-01-01,1,1001,0,445838.0,445838,0,0,NaN,0.0,1,1,0,Benin,Large,Televisions,LG Televisions Model-120,Televisions,LG,445838,332772,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
1,2021-01-01,1,1002,2,500410.0,500410,0,0,NaN,1000820.0,6,4,0,Benin,Large,Mobile Phones,Tecno Mobile Model-199,Mobile Phones,Tecno,500410,346208,1,6,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
2,2021-01-01,1,1003,2,399365.0,399365,0,0,NaN,798730.0,2,0,1,Benin,Large,Mobile Phones,Tecno Mobile Model-905,Mobile Phones,Tecno,399365,244124,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
3,2021-01-01,1,1004,4,305796.0,305796,0,0,NaN,1223184.0,10,6,0,Benin,Large,Mobile Phones,Infinix Mobile Model-121,Mobile Phones,Infinix,305796,221242,1,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN
4,2021-01-01,1,1005,5,462752.0,462752,0,0,NaN,2313760.0,28,23,0,Benin,Large,Mobile Phones,Samsung Mobile Model-781,Mobile Phones,Samsung,462752,326219,1,24,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,1,1,Friday,False,True,False,Dry,30.6,3.7,Rainy,NaN,NaN,NaN


In [12]:
df["promo_type"]

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
          ... 
1227235    NaN
1227236    NaN
1227237    NaN
1227238    NaN
1227239    NaN
Name: promo_type, Length: 1227240, dtype: object

In [13]:
df_filtered = df[df["promo_type"].notna()]

In [14]:
df_filtered

,date,store_id,product_id,units_sold,price,regular_price,discount_pct,promo_flag,promo_type,revenue,starting_inventory,ending_inventory,stockout_occurred,city,store_size,category,product_name,category_product,brand,regular_price_product,cost_price,is_seasonal,warranty_months,store_name,city_store,area,region,store_type,store_size_store,year,month,day,day_of_week,is_weekend,is_holiday,is_payday,season,temperature_c,rainfall_mm,weather_condition,promo_type_promo,discount_pct_promo,promo_flag_promo
71409,2021-03-27,1,1010,0,722843.00,722843,0,1,Bundle,0.00,1,1,0,Benin,Large,Laptops & Computers,Acer Laptops Model-134,Laptops & Computers,Acer,722843,505292,0,24,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,3,27,Saturday,True,False,False,Early Rainy,29.3,1.8,Cloudy,Bundle,0.0,1.0
71454,2021-03-27,1,1055,1,839944.00,839944,0,1,Bundle,839944.00,3,2,0,Benin,Large,Laptops & Computers,Dell Laptops Model-669,Laptops & Computers,Dell,839944,570270,0,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2021,3,27,Saturday,True,False,False,Early Rainy,29.3,1.8,Cloudy,Bundle,0.0,1.0
71538,2021-03-27,2,1019,1,328325.40,364806,10,1,Discount,328325.40,3,2,0,Ekpoma,Medium,Televisions,LG Televisions Model-132,Televisions,LG,364806,280903,1,12,SunnyBest Ekpoma,Ekpoma,Esan West,Edo Central,High Street,Medium,2021,3,27,Saturday,True,False,False,Early Rainy,28.3,2.1,Cloudy,Discount,10.0,1.0
71703,2021-03-27,3,1064,1,401451.00,401451,0,1,Free Accessory,401451.00,2,1,0,Auchi,Medium,Laptops & Computers,Asus Laptops Model-202,Laptops & Computers,Asus,401451,258198,0,6,SunnyBest Auchi,Auchi,Etsako West,Edo North,High Street,Medium,2021,3,27,Saturday,True,False,False,Early Rainy,30.4,1.2,Cloudy,Free Accessory,0.0,1.0
71713,2021-03-27,3,1074,0,584963.00,584963,0,1,Free Accessory,0.00,1,1,0,Auchi,Medium,Refrigerators,Haier Thermocool Refrigerators Model-456,Refrigerators,Haier Thermocool,584963,371241,1,12,SunnyBest Auchi,Auchi,Etsako West,Edo North,High Street,Medium,2021,3,27,Saturday,True,False,False,Early Rainy,30.4,1.2,Cloudy,Free Accessory,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1221478,2024-12-25,1,1119,2,625245.35,658153,5,1,Discount,1250490.70,4,2,0,Benin,Large,Laptops & Computers,Dell Laptops Model-870,Laptops & Computers,Dell,658153,463380,0,12,SunnyBest Benin Main,Benin,Oredo,Edo South,Mall,Large,2024,12,25,Wednesday,False,True,True,Dry,28.9,1.4,Cloudy,Discount,5.0,1.0
1221787,2024-12-25,4,1068,4,304283.70,338093,10,1,Discount,1217134.80,10,6,0,Irrua,Small,Mobile Phones,Itel Mobile Model-740,Mobile Phones,Itel,338093,259383,1,6,SunnyBest Irrua,Irrua,Esan Central,Edo Central,Plaza,Small,2024,12,25,Wednesday,False,True,True,Dry,26.1,0.0,Sunny,Discount,10.0,1.0
1221865,2024-12-25,5,1026,3,49885.65,58689,15,1,Discount,149656.95,9,6,0,Igueben,Small,Small Appliances,Philips Small Model-745,Small Appliances,Philips,58689,44698,0,6,SunnyBest Igueben,Igueben,Igueben,Edo Central,High Street,Small,2024,12,25,Wednesday,False,True,True,Dry,26.1,2.7,Cloudy,Discount,15.0,1.0
1222095,2024-12-25,7,1016,1,54976.00,54976,0,1,Free Accessory,54976.00,4,3,0,Ogwa,Small,Small Appliances,Binatone Small Model-164,Small Appliances,Binatone,54976,36243,0,6,SunnyBest Ogwa,Ogwa,Esan West,Edo Central,High Street,Small,2024,12,25,Wednesday,False,True,True,Dry,27.3,0.5,Cloudy,Free Accessory,0.0,1.0


In [ ]:
['date', 'store_id', 'product_id', 'units_sold', 'price',
       'regular_price', 'discount_pct', 'promo_flag', 'promo_type', 'revenue',
       'starting_inventory', 'ending_inventory', 'stockout_occurred', 'city',
       'store_size', 'category', 'product_name', 'category_product', 'brand',
       'regular_price_product', 'cost_price', 'is_seasonal', 'warranty_months',
       'store_name', 'city_store', 'area', 'region', 'store_type',
       'store_size_store', 'year', 'month', 'day', 'day_of_week', 'is_weekend',
       'is_holiday', 'is_payday', 'season', 'temperature_c', 'rainfall_mm',
       'weather_condition', 'promo_type_promo', 'discount_pct_promo',
       'promo_flag_promo']

In [17]:
df[unit_sold].plot()

NameError: name 'unit_sold' is not defined